# Notebook 18 — Seed robustness and paired ensemble diagnostics

**Development-only.** Notebook 17 found only a marginal difference between Random Forest and equal-weight Logistic L2 + Random Forest soft voting. This notebook repeats the same chronological experiment with multiple random seeds, quantifies paired fold differences, and inspects neural training stability. It does **not** read the final holdout or write prospective predictions.

Repeated seeds and folds are correlated: reported differences are **descriptive**, not independent statistical tests.

In [ ]:
from pathlib import Path
import sys
ROOT=Path.cwd().resolve()
if ROOT.name=='notebooks': ROOT=ROOT.parent
if not (ROOT/'src'/'stock_ml_dynamic').exists(): raise RuntimeError('Open notebook from stock_ml_dynamic project')
sys.path.insert(0,str(ROOT/'src'))
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from stock_ml_dynamic.pooled_models import load_master
from stock_ml_dynamic.robustness_diagnostics import run_robustness
required=('master_labeled.parquet','latest_inference.parquet','transformers.joblib','manifest.json')
candidates=[ROOT/'artifacts'/'master_tables'/'1d',ROOT/'notebooks'/'artifacts'/'master_tables'/'1d']
master_dir=next((p for p in candidates if all((p/f).exists() for f in required)),None)
if master_dir is None:raise FileNotFoundError('Notebook 03 master table artifacts are required')
master,latest,bundle,manifest=load_master(master_dir)
print('Rows:',len(master),'symbols:',master.symbol.nunique())

## Run configuration

Start with the smoke test. Full repeated-seed experiments can be computationally expensive, particularly the custom residual network. No tuning against the historical holdout.

In [ ]:
SMOKE_TEST=True
SEEDS=(42,52) if SMOKE_TEST else (42,52,62,72,82)
N_FOLDS=2 if SMOKE_TEST else 5
NEURAL_EPOCHS=8 if SMOKE_TEST else 35
MAX_TRAIN_ROWS=60000
TREE_TRAIN_ROWS=45000
OUTPUT=ROOT/'artifacts'/'robustness_diagnostics'/'1d'/('smoke' if SMOKE_TEST else 'full')
scores,summary,paired,development_end=run_robustness(master,seeds=SEEDS,n_folds=N_FOLDS,neural_epochs=NEURAL_EPOCHS,max_train_rows=MAX_TRAIN_ROWS,tree_train_rows=TREE_TRAIN_ROWS,output_dir=OUTPUT)
print('Development ends:',development_end)
print('Final holdout and prospective ledger untouched')

## Aggregate performance and paired fold comparisons

A positive paired delta means a model exceeded Random Forest in the same seed and chronological fold.

In [ ]:
display(summary)
display(paired)
fig,ax=plt.subplots(figsize=(9,4))
ax.bar(summary.model,summary.roc_auc_mean)
ax.set_ylabel('Mean macro OVR ROC-AUC')
ax.set_ylim(.48,max(.60,float(summary.roc_auc_mean.max())+.02))
ax.tick_params(axis='x',rotation=30)
plt.tight_layout();plt.show()

## Paired performance variation

Inspect whether ensemble improvements are consistent across both seeds and market periods.

In [ ]:
wide=scores.pivot(index=['seed','fold'],columns='model',values='roc_auc')
delta=wide['soft_vote_lr_rf']-wide['random_forest']
display(delta.rename('soft_vote_minus_rf').reset_index())
fig,ax=plt.subplots(figsize=(9,4))
ax.axhline(0,color='gray',linewidth=1)
ax.plot(range(1,len(delta)+1),delta.to_numpy(),marker='o')
ax.set(xlabel='Seed × fold run',ylabel='ROC-AUC difference',title='Soft voting minus Random Forest')
plt.tight_layout();plt.show()

## Neural network training stability

Validation curves from each seed/fold are saved in the run-specific directories. Early stopping uses only chronological training subdivisions.

In [ ]:
fig,ax=plt.subplots(figsize=(9,4))
for seed in SEEDS:
    f=OUTPUT/f'seed_{seed}'/'residual_training_history.csv'
    if f.exists():
        history=pd.read_csv(f)
        for fold,g in history.groupby('fold'):
            ax.plot(g.epoch,g.val_loss,label=f'seed {seed} fold {fold}',alpha=.75)
ax.set(xlabel='Epoch',ylabel='Chronological validation loss',title='Residual MLP training stability')
ax.legend(fontsize=8)
plt.tight_layout();plt.show()

## Interpretation and next steps

- Prefer an ensemble only if its paired improvement is persistent and material relative to its inference/maintenance cost.
- If the residual MLP remains weaker, investigate feature scaling, architecture width, and learning-rate schedules **within development folds**, rather than consulting the closed holdout.
- Repeated folds are not independent; do not interpret a positive average as proof of statistical significance.
- Model selection and deployment changes must be versioned separately from the frozen prospective Logistic L2 baseline.
- The next notebook can investigate nested probability calibration and leakage-safe stacking, but it must fit calibration and stacking exclusively on earlier chronological observations.